# Exercise 01 — Async API Calls

One of the most common things you'll do as an AI Engineer is call multiple APIs at the same time. If you do it sequentially, you're just sitting there waiting. With `asyncio.gather`, all the calls go out at once and you collect the results when they come back.

This exercise uses [JSONPlaceholder](https://jsonplaceholder.typicode.com), a free fake REST API. No keys needed.

**What you'll practice:**
- Writing `async def` functions
- Using `httpx.AsyncClient` for HTTP
- Running concurrent calls with `asyncio.gather`

In [ ]:
!pip install httpx -q

In [ ]:
import asyncio
import httpx

BASE_URL = "https://jsonplaceholder.typicode.com"
POST_IDS = [1, 2, 3, 4, 5]


async def fetch_post(client: httpx.AsyncClient, post_id: int) -> dict:
    response = await client.get(f"{BASE_URL}/posts/{post_id}")
    response.raise_for_status()
    return response.json()


async def fetch_all_posts(post_ids: list[int]) -> list[dict]:
    async with httpx.AsyncClient() as client:
        tasks = [fetch_post(client, pid) for pid in post_ids]
        results = await asyncio.gather(*tasks)
    return list(results)


# In Colab the event loop is already running, so we use await directly
posts = await fetch_all_posts(POST_IDS)

for post in posts:
    print(f"[{post['id']}] {post['title'][:60]}")

## Check your understanding

- What happens if you replace `asyncio.gather` with a regular `for` loop of `await` calls? Try it and compare the feel.
- What does `raise_for_status()` do? When would you want to catch the error instead of raising it?
- Why do we open a single `AsyncClient` and reuse it for all requests instead of creating one per call?

In [ ]:
# Try it: rewrite fetch_all_posts using a for loop instead of gather
# Then time both versions with %%time to see the difference

async def fetch_all_posts_sequential(post_ids: list[int]) -> list[dict]:
    results = []
    async with httpx.AsyncClient() as client:
        for pid in post_ids:
            post = await fetch_post(client, pid)
            results.append(post)
    return results